# Modelado XGBoost (2018–2024)

Este notebook forma parte del pipeline de ciencia de datos del proyecto **crash-severity-predictor**.

El objetivo es desarrollar, entrenar y evaluar un modelo **XGBoost** para la predicción de severidad en hechos de tránsito a partir del dataset procesado durante las fases de EDA y ETL. Esta implementación constituye la segunda iteración dentro del conjunto de modelos candidatos del proyecto.



In [1]:
# -- Importaciones ----------------------------------------------
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score, accuracy_score,
                             roc_curve)
import plotly.graph_objects as go
import plotly.express as px
import json, os
import warnings
warnings.filterwarnings('ignore')

print('✓ Librerías cargadas correctamente')

✓ Librerías cargadas correctamente


## 1. Carga de datos

In [2]:
# -- Carga ----------------------------------------------
train = pd.read_parquet('../data/clean/train.parquet')
test  = pd.read_parquet('../data/clean/test.parquet')

FEATURES = ['tipo_eve','tipo_veh','g_hora_5','dia_sem_ocu',
            'sexo_per','edad_quinquenales','mayor_menor','depto_ocu']
TARGET = 'fall_les'

X_train = train[FEATURES]
y_train = train[TARGET] - 1  # XGBoost requiere clases 0 y 1
X_test  = test[FEATURES]
y_test  = test[TARGET] - 1

print(f'Train : {X_train.shape}')
print(f'Test  : {X_test.shape}')
print(f'\nClases: {sorted(y_train.unique())}')

Train : (57553, 8)
Test  : (14389, 8)

Clases: [np.int64(0), np.int64(1)]


## 2. Entrenamiento XGBoost

In [3]:
# -- Entrenamiento ----------------------------------------------
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='auc',
    verbosity=0
)

xgb.fit(X_train, y_train)
print(f'✓ Modelo entrenado correctamente')
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

✓ Modelo entrenado correctamente
scale_pos_weight: 0.24


## 3. Evaluación del modelo

In [4]:
# -- Predicciones ----------------------------------------------
y_pred  = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average='weighted')
auc = roc_auc_score(y_test, y_proba)
cm  = confusion_matrix(y_test, y_pred)

print('=== XGBoost ===')
print(f'Accuracy : {acc:.4f}')
print(f'F1-Score : {f1:.4f}')
print(f'ROC-AUC  : {auc:.4f}')
print(f'\n{classification_report(y_test, y_pred, target_names=["Fallecido","Lesionado"])}')
print(f'Matriz de confusión:')
print(cm)

=== XGBoost ===
Accuracy : 0.6718
F1-Score : 0.7053
ROC-AUC  : 0.7392

              precision    recall  f1-score   support

   Fallecido       0.33      0.68      0.44      2748
   Lesionado       0.90      0.67      0.77     11641

    accuracy                           0.67     14389
   macro avg       0.61      0.67      0.60     14389
weighted avg       0.79      0.67      0.71     14389

Matriz de confusión:
[[1856  892]
 [3830 7811]]


In [5]:
# -- Visualizaciones ----------------------------------------------
from plotly.subplots import make_subplots

# Métricas
fig_metricas = go.Figure(go.Bar(
    x=['Accuracy', 'F1-Score', 'ROC-AUC'],
    y=[acc, f1, auc],
    text=[f'{acc:.4f}', f'{f1:.4f}', f'{auc:.4f}'],
    textposition='auto',
    marker_color=['#F4A261', '#E76F51', '#2EC4B6'],
    width=0.4
))
fig_metricas.update_layout(
    title='Métricas de evaluación : XGBoost',
    yaxis=dict(range=[0, 1], title='Valor'),
    xaxis_title='Métrica',
    height=400,
    template='plotly_white'
)
fig_metricas.show()

In [6]:
# -- Matriz de confusión ----------------------------------------------
fig_cm = px.imshow(
    cm,
    labels=dict(x='Predicción', y='Real', color='Cantidad'),
    x=['Fallecido', 'Lesionado'],
    y=['Fallecido', 'Lesionado'],
    text_auto=True,
    color_continuous_scale='Oranges',
    title='Matriz de confusión : XGBoost'
)
fig_cm.update_layout(height=400, template='plotly_white')
fig_cm.show()

In [7]:
# -- Curva ROC ----------------------------------------------
fpr, tpr, _ = roc_curve(y_test, y_proba)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines',
    name=f'XGBoost (AUC = {auc:.4f})',
    line=dict(color='#F4A261', width=2.5)
))
fig_roc.add_trace(go.Scatter(
    x=[0,1], y=[0,1],
    mode='lines',
    name='Baseline (AUC = 0.5)',
    line=dict(color='gray', width=1.5, dash='dash')
))
fig_roc.update_layout(
    title='Curva ROC : XGBoost',
    xaxis_title='Tasa de Falsos Positivos',
    yaxis_title='Tasa de Verdaderos Positivos',
    height=450,
    template='plotly_white',
    legend=dict(x=0.6, y=0.1)
)
fig_roc.show()

In [8]:
# -- Importancia de features ----------------------------------------------
importancias_xgb = pd.Series(
    xgb.feature_importances_, index=FEATURES
).sort_values()

LABELS = {
    'tipo_eve'         : 'Tipo de evento',
    'tipo_veh'         : 'Tipo de vehículo',
    'g_hora_5'         : 'Grupo horario',
    'dia_sem_ocu'      : 'Día de la semana',
    'sexo_per'         : 'Sexo',
    'edad_quinquenales': 'Grupo de edad',
    'mayor_menor'      : 'Mayor / Menor edad',
    'depto_ocu'        : 'Departamento'
}

fig_imp = go.Figure(go.Bar(
    x=importancias_xgb.values,
    y=[LABELS[f] for f in importancias_xgb.index],
    orientation='h',
    marker_color='#F4A261',
    text=[f'{v:.4f}' for v in importancias_xgb.values],
    textposition='auto',
    textfont=dict(size=11)
))
fig_imp.update_layout(
    title='Importancia de features : XGBoost',
    xaxis=dict(title='Importancia (F-score)', range=[0, max(importancias_xgb.values) * 1.25]),
    yaxis_title='Feature',
    height=450,
    template='plotly_white'
)
fig_imp.show()

### Hallazgo — Importancia de features XGBoost

| Feature | Importancia |
|---|---|
| Sexo | 30.38% : más predictora |
| Tipo de evento | 17.37% |
| Departamento | 12.04% |
| Mayor / Menor edad | 10.59% |
| Tipo de vehículo | 9.17% |
| Grupo de edad | 8.87% |
| Grupo horario | 6.57% |
| Día de la semana | 5.02% : menos predictora |

Sin SMOTE, XGBoost distribuye mejor la importancia entre features.
Sexo baja de 62% a 30% — modelo más robusto y menos sesgado.

In [9]:
# -- Guardar resultados ----------------------------------------------
resultados_xgb = {
    'modelo'             : 'XGBoost',
    'accuracy'           : round(acc, 4),
    'f1_score'           : round(f1, 4),
    'roc_auc'            : round(auc, 4),
    'precision_fallecido': 0.32,
    'recall_fallecido'   : 0.66,
}

with open('../data/models/resultados_xgb.json', 'w') as f:
    json.dump(resultados_xgb, f, indent=2)

print('✓ Resultados guardados en data/models/resultados_xgb.json')
print(f'\nResumen XGBoost:')
for k, v in resultados_xgb.items():
    print(f'  {k:<25} {v}')

✓ Resultados guardados en data/models/resultados_xgb.json

Resumen XGBoost:
  modelo                    XGBoost
  accuracy                  0.6718
  f1_score                  0.7053
  roc_auc                   0.7392
  precision_fallecido       0.32
  recall_fallecido          0.66


In [10]:
# -- Guardar modelo entrenado ----------------------------------------------
import joblib
import os

os.makedirs('../data/models', exist_ok=True)
joblib.dump(xgb, '../data/models/xgboost.pkl')
print('✓ Modelo guardado en data/models/xgboost.pkl')

✓ Modelo guardado en data/models/xgboost.pkl


## 4. Resumen del modelo

| Métrica | Valor |
|---|---|
| Accuracy | 67.18% |
| F1-Score (weighted) | 70.53% |
| ROC-AUC | 73.92% |
| Precision Fallecido | 33% |
| Recall Fallecido | 68% |

**Conclusiones:**
- ROC-AUC de 0.7392 — mejor poder discriminativo sin SMOTE
- Recall Fallecido de 68% — detecta 2 de cada 3 casos fatales
- Importancia de features más balanceada — Sexo baja de 62% a 30%
- Departamento y Tipo de evento ganan peso predictivo real